# 🏁 Prueba Final Offline — UIS Grand Prix Foundationals Models

1. Coloca todos los archivos `*.zip` exportados por los estudiantes dentro de `prueba_final/entregas/`.
2. Ejecuta la **Celda 1** para evaluar a todos en `B1, B2, B3`, calcular el **Ranking F1** y guardar `resultados_finales.json` + `trayectorias_finales.json`.
3. Ejecuta la **Celda 2** en este PC para proyectar en Pygame la carrera conjunta (**Modo Fantasma** y **Modo Legendario**).

In [7]:
# Celda 1: Cargar todas las entregas (.zip), evaluar en B1-B3 y guardar resultados + trayectorias
import json
import os
import shutil
import sys
import tempfile
import zipfile
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "prueba_final":
    ROOT = ROOT.parent
PRUEBA_DIR = ROOT / "prueba_final"
ENTREGAS_DIR = PRUEBA_DIR / "entregas"
ENTREGAS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "backend_server"))
sys.path.insert(0, str(ROOT / "participante"))

import config_estudiante as cfg
import juego_pistas
import motor_qwen
import server as srv

zips = sorted(ENTREGAS_DIR.glob("*.zip"))
jsons = sorted(ENTREGAS_DIR.glob("*.json"))
entregas = zips + jsons
print(f"Entregas encontradas ({len(entregas)}): {[e.name for e in entregas]}")
if not entregas:
    raise RuntimeError(f"Coloca archivos .zip o .json en {ENTREGAS_DIR}")

motor = motor_qwen.motor_global
motor.inicializar()

ranking_data = {}
trayectorias_b = {"B1": [], "B2": [], "B3": []}

for ent_path in entregas:
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp = Path(tmpdir)
        if ent_path.suffix.lower() == ".zip":
            with zipfile.ZipFile(ent_path, "r") as zf:
                zf.extractall(tmp)
            manifest = json.loads((tmp / "manifest.json").read_text(encoding="utf-8"))
            jugadas = json.loads((tmp / "mis_jugadas.json").read_text(encoding="utf-8"))
        else:
            raw_json = json.loads(ent_path.read_text(encoding="utf-8"))
            if isinstance(raw_json, list):
                manifest = {
                    "student_id": ent_path.stem,
                    "student_name": ent_path.stem,
                    "estrategia": "ICL",
                }
                jugadas = raw_json
            elif isinstance(raw_json, dict):
                manifest = dict(raw_json.get("manifest", raw_json))
                manifest.setdefault("student_id", ent_path.stem)
                manifest.setdefault("student_name", manifest["student_id"])
                manifest.setdefault("estrategia", "ICL")
                jugadas = raw_json.get("mis_jugadas", raw_json.get("jugadas", []))
            else:
                print(f"Saltando {ent_path.name}: formato JSON no reconocido.")
                continue

        sid = motor_qwen.sanitizar_student_id(manifest["student_id"])
        sname = manifest.get("student_name", sid)
        estrat = str(manifest["estrategia"]).upper().strip()
        aname = manifest.get("adapter_name") or "default"

        # Copiar adaptador LoRA temporalmente a adapters_cache si aplica
        adapter_src = tmp / "adapter"
        if estrat in ("LORA", "LORA_ICL") and (adapter_src / "adapter_config.json").exists():
            akey = motor._adapter_key(sid, aname)
            dest_dir = motor._ruta_adaptador(akey)
            dest_dir.mkdir(parents=True, exist_ok=True)
            for f in adapter_src.iterdir():
                if f.is_file():
                    shutil.copy2(f, dest_dir / f.name)

        # Aplicar configuración exacta del estudiante
        cfg.ESTRATEGIA = estrat
        cfg.CONFIG_DATOS.update(manifest.get("CONFIG_DATOS", {}))
        cfg.CONFIG_ICL.update(manifest.get("CONFIG_ICL", {}))
        cfg.CONFIG_LORA.update(manifest.get("CONFIG_LORA", {}))

        class LocalBridge:
            def decidir_accion(self, estrategia, estado_actual, historial, jugadas_referencia=None, config_icl=None, config_datos=None, velocidad=None, cp_dist=None, cp_ang=None, adapter_name=None):
                ci = config_icl or {}
                cd = config_datos or {}
                r = motor.decidir_accion(
                    student_id=sid,
                    estrategia=estrategia,
                    estado_actual=estado_actual,
                    historial=historial,
                    jugadas_referencia=jugadas_referencia,
                    k_shots=int(ci.get("k_shots", 4)),
                    seleccion=str(ci.get("seleccion", "knn")),
                    pesos_sensores=ci.get("pesos_sensores"),
                    temperature=float(ci.get("temperature", 0.0)),
                    top_p=float(ci.get("top_p", 0.9)),
                    precision_sensores=int(cd.get("precision_sensores", 50)),
                    ventana_historia=int(cd.get("ventana_historia", 5)),
                    velocidad=velocidad,
                    cp_dist=cp_dist,
                    cp_ang=cp_ang,
                    adapter_name=aname,
                )
                r["rtt_ms"] = r["latencia_gpu_ms"]
                return r

        lb = LocalBridge()
        print(f"\nEvaluando: {sname} ({estrat}) | jugadas={len(jugadas)}...")
        res_pistas = {}
        for p in juego_pistas.PISTAS_TEST:
            r = juego_pistas.simular_pista_autonoma_headless(lb, p, jugadas)
            res_pistas[p] = r
            trayectorias_b[p].append({
                "entry_id": f"{sid}__{estrat}",
                "student_name": sname,
                "estrategia": estrat,
                "trayectoria": r.get("trayectoria", []),
            })
            print(f"  {p}: {r['progreso_pct']:.1f}% | t={r['tiempo_fisico_s']:.1f}s | choques={r['choques']} | gpu={r['latencia_promedio_ms']:.1f}ms")

        n_ej = len(jugadas) if estrat == "ICL" else min(len(jugadas), int(cfg.CONFIG_LORA.get("num_ejemplos", 200)))
        entry_key = f"{sid}__{estrat}"
        ranking_data[entry_key] = {
            "entry_id": entry_key,
            "student_id": sid,
            "student_name": sname,
            "estrategia": estrat,
            "num_ejemplos": n_ej,
            "progreso_promedio_pct": round(sum(res_pistas[p]["progreso_pct"] for p in ("B1", "B2", "B3")) / 3.0, 2),
            "pistas_completadas": sum(1 for p in ("B1", "B2", "B3") if res_pistas[p]["completada"]),
            "choques_totales": sum(res_pistas[p]["choques"] for p in ("B1", "B2", "B3")),
            "tiempo_fisico_total_s": round(sum(res_pistas[p]["tiempo_fisico_s"] for p in ("B1", "B2", "B3")), 3),
            "latencia_promedio_ms": round(sum(res_pistas[p]["latencia_promedio_ms"] for p in ("B1", "B2", "B3")) / 3.0, 2),
        }

general, top_c, top_v, top_d, top_l = srv._calcular_rankings_f1(ranking_data)
(PRUEBA_DIR / "resultados_finales.json").write_text(json.dumps(general, indent=2, ensure_ascii=False), encoding="utf-8")
(PRUEBA_DIR / "trayectorias_finales.json").write_text(json.dumps({"pistas": trayectorias_b}, ensure_ascii=False), encoding="utf-8")

print("\n" + "=" * 75)
print("🏆 CAMPEONATO GENERAL F1 (55% Carrera | 20% Vel | 15% Few-Shot | 10% Limpieza)")
print("=" * 75)
for i, r in enumerate(general, 1):
    print(f"#{i:2d} | {r['student_name']:16s} | {r['estrategia']:8s} | Pts={r['puntos_f1']:5.2f} | B={r['progreso_promedio_pct']:5.1f}% | Lat={r['latencia_promedio_ms']:5.1f}ms | Ej={r['num_ejemplos']}")

ModuleNotFoundError: No module named 'config_estudiante'

In [ ]:
# Celda 2: Proyectar en este PC la Carrera Final (Modo Fantasma y Modo Legendario con M)
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "prueba_final":
    ROOT = ROOT.parent
PRUEBA_DIR = ROOT / "prueba_final"
sys.path.insert(0, str(ROOT / "participante"))

import bridge
import juego_pistas

tray_data = json.loads((PRUEBA_DIR / "trayectorias_finales.json").read_text(encoding="utf-8"))

class OfflineBridge(bridge.ServerBridge):
    def __init__(self):
        pass
    def verificar_salud(self):
        return {"status": "ok", "gpu_nombre": "Offline"}
    def listar_adaptadores(self):
        return []
    def obtener_trayectorias_b(self):
        return tray_data

app = juego_pistas.JuegoPygame(OfflineBridge())
app._pedir_clave_docente = lambda: True
try:
    app._pantalla_carrera_todos()
finally:
    app.pygame.quit()